# Week 8 Community Contribution - Adnan Gobeljic

This notebook builds a **lightweight autonomous deal-hunting workflow** inspired by Week 8 (Days 1-5):

- **Day 1:** specialist-style pricing agent
- **Day 2:** frontier/RAG-style retrieval estimator + ensemble
- **Day 3:** scanner + messenger behavior
- **Day 4:** autonomous planner that picks the best opportunity
- **Day 5:** optional Gradio UI wrapper

The implementation is intentionally compact and runnable with minimal setup while still reflecting the multi-agent architecture from the course and community submissions.

## Setup notes

- This notebook is designed to run **without mandatory external APIs**.
- If `OPENAI_API_KEY` is set, an optional LLM-based deal judge can be used.
- If `gradio` is installed, you can launch a simple UI in the last section.

This makes the contribution practical for quick local testing while still matching the Week 8 agentic flow.

In [ ]:
import json
import math
import os
import re
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime
from typing import Dict, List, Optional, Tuple

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except Exception:
    pass

print("Environment ready")

## 1) ScannerAgent-style parsing (Day 3)

We simulate RSS/raw deal feed text, then parse product description, listed price, and URL.

The scanner keeps higher-quality entries (clear price + URL + detailed description).

In [ ]:
@dataclass
class Deal:
    product_description: str
    listed_price: float
    url: str


RAW_FEED = [
    "Sony WH-1000XM5 wireless noise-canceling headphones, 30-hour battery, premium ANC. Deal price $249 at https://deals.example.com/sony-xm5",
    "Apple AirPods Pro 2 (USB-C), active noise cancellation, MagSafe charging case, now $199 on https://deals.example.com/airpods-pro-2",
    "Dell XPS 13 laptop, Intel Ultra 7, 16GB RAM, 512GB SSD, listed for $899: https://deals.example.com/xps13",
    "Mechanical keyboard with hot-swappable switches and RGB, reduced by $40 at https://deals.example.com/mech-kb",
    "Samsung 55 inch 4K QLED Smart TV with HDR support and voice assistant built in. Offer $499 at https://deals.example.com/samsung-qled",
    "Portable USB-C hub 8-in-1 with HDMI, ethernet and power delivery for $39.99 link https://deals.example.com/usb-hub",
]


def parse_deal(text: str) -> Optional[Deal]:
    price_match = re.search(r"\$(\d+(?:\.\d{1,2})?)", text)
    url_match = re.search(r"https?://\S+", text)

    # Reject entries without unambiguous listed price and URL.
    if not price_match or not url_match:
        return None

    listed_price = float(price_match.group(1))
    url = url_match.group(0).rstrip(".,)")

    cleaned = text.replace(url_match.group(0), "").strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    # Ignore entries that mostly mention discount amount instead of true price.
    if "off" in cleaned.lower() and "now" not in cleaned.lower() and "deal price" not in cleaned.lower() and "listed for" not in cleaned.lower():
        return None

    return Deal(product_description=cleaned, listed_price=listed_price, url=url)


def scan_deals(raw_items: List[str], min_chars: int = 60, top_k: int = 5) -> List[Deal]:
    parsed = [deal for deal in (parse_deal(item) for item in raw_items) if deal is not None]
    parsed = [deal for deal in parsed if len(deal.product_description) >= min_chars]
    parsed.sort(key=lambda d: len(d.product_description), reverse=True)
    return parsed[:top_k]


scanned_deals = scan_deals(RAW_FEED)
print(f"Scanner selected {len(scanned_deals)} deals")
for i, deal in enumerate(scanned_deals, start=1):
    print(f"{i}. ${deal.listed_price:.2f} | {deal.url}")
    print(f"   {deal.product_description[:110]}...")

## 2) Frontier-style retrieval estimate (Day 2)

Instead of a full vector DB, this notebook uses a tiny local catalog and cosine similarity over token counts to emulate retrieval context for valuation.

In [ ]:
REFERENCE_CATALOG = [
    {"description": "Sony WH-1000XM5 wireless headphones active noise cancelling", "fair_price": 329.0},
    {"description": "Apple AirPods Pro 2 with MagSafe case", "fair_price": 249.0},
    {"description": "Dell XPS 13 ultrabook 16GB RAM 512GB SSD", "fair_price": 1099.0},
    {"description": "Samsung 55 inch QLED 4K smart television", "fair_price": 699.0},
    {"description": "USB-C 8 in 1 hub HDMI ethernet PD", "fair_price": 59.0},
    {"description": "Mechanical keyboard RGB hot swap switches", "fair_price": 119.0},
]


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def to_vector(text: str) -> Counter:
    return Counter(tokenize(text))


def cosine_similarity(a: Counter, b: Counter) -> float:
    if not a or not b:
        return 0.0
    common = set(a).intersection(b)
    dot = sum(a[token] * b[token] for token in common)
    norm_a = math.sqrt(sum(v * v for v in a.values()))
    norm_b = math.sqrt(sum(v * v for v in b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def find_similars(description: str, n_results: int = 3) -> List[Tuple[Dict, float]]:
    query_vec = to_vector(description)
    scored = []
    for entry in REFERENCE_CATALOG:
        score = cosine_similarity(query_vec, to_vector(entry["description"]))
        scored.append((entry, score))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:n_results]


def frontier_estimate(description: str) -> Tuple[float, List[Tuple[Dict, float]]]:
    similars = find_similars(description)
    weights = [score + 0.05 for _, score in similars]
    prices = [entry["fair_price"] for entry, _ in similars]
    weighted = sum(w * p for w, p in zip(weights, prices)) / sum(weights)
    return round(weighted, 2), similars


example_frontier_price, example_context = frontier_estimate(scanned_deals[0].product_description)
print("Frontier estimate:", example_frontier_price)
print("Top retrieval context:")
for entry, score in example_context:
    print(f"- sim={score:.3f} | ${entry['fair_price']:.2f} | {entry['description']}")

## 3) Specialist estimate + Ensemble (Days 1 and 2)

- **Specialist-style** estimator uses product-domain heuristics.
- **Frontier-style** estimator uses retrieval context.
- **Ensemble** combines both and returns a confidence score.

In [ ]:
KEYWORD_BASELINES = {
    "headphones": 280.0,
    "airpods": 220.0,
    "laptop": 980.0,
    "xps": 1100.0,
    "tv": 650.0,
    "keyboard": 110.0,
    "hub": 55.0,
}


def specialist_estimate(description: str) -> float:
    text = description.lower()
    matches = [price for key, price in KEYWORD_BASELINES.items() if key in text]
    if matches:
        return round(sum(matches) / len(matches), 2)

    # Fallback heuristic: longer descriptions tend to be pricier items in this toy example.
    token_count = len(tokenize(description))
    heuristic = 20 + token_count * 8
    return round(float(heuristic), 2)


def ensemble_estimate(description: str) -> Dict:
    specialist = specialist_estimate(description)
    frontier, similars = frontier_estimate(description)

    estimate = round(0.55 * specialist + 0.45 * frontier, 2)
    disagreement = abs(specialist - frontier)
    confidence = max(0.2, 1.0 - (disagreement / max(max(specialist, frontier), 1.0)))

    return {
        "estimate": estimate,
        "specialist": specialist,
        "frontier": frontier,
        "confidence": round(confidence, 3),
        "similars": similars,
    }


sample_eval = ensemble_estimate(scanned_deals[0].product_description)
print(json.dumps({k: v for k, v in sample_eval.items() if k != "similars"}, indent=2))

## 4) Autonomous planning + messaging (Days 3 and 4)

The planner scores each scanned deal using ensemble estimates, computes discount (`estimated_value - listed_price`), and selects the best opportunity above a minimum threshold.

In [5]:
@dataclass
class Opportunity:
    deal: Deal
    estimate: float
    specialist: float
    frontier: float
    confidence: float
    discount: float


def evaluate_deal(deal: Deal) -> Opportunity:
    result = ensemble_estimate(deal.product_description)
    discount = round(result["estimate"] - deal.listed_price, 2)
    return Opportunity(
        deal=deal,
        estimate=result["estimate"],
        specialist=result["specialist"],
        frontier=result["frontier"],
        confidence=result["confidence"],
        discount=discount,
    )


def rank_opportunities(deals: List[Deal]) -> List[Opportunity]:
    opportunities = [evaluate_deal(deal) for deal in deals]
    opportunities.sort(key=lambda o: (o.discount, o.confidence), reverse=True)
    return opportunities


def choose_best(opportunities: List[Opportunity], min_discount: float = 40.0) -> Optional[Opportunity]:
    for opp in opportunities:
        if opp.discount >= min_discount:
            return opp
    return None


def notify(opportunity: Opportunity) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    message = (
        f"[{timestamp}] DEAL ALERT\\n"
        f"Price: ${opportunity.deal.listed_price:.2f}\\n"
        f"Estimated value: ${opportunity.estimate:.2f}\\n"
        f"Potential savings: ${opportunity.discount:.2f}\\n"
        f"URL: {opportunity.deal.url}"
    )
    print(message)


def run_cycle(raw_items: List[str], min_discount: float = 40.0):
    deals = scan_deals(raw_items)
    opportunities = rank_opportunities(deals)
    best = choose_best(opportunities, min_discount=min_discount)
    return deals, opportunities, best

## 5) Run full cycle and inspect output

In [ ]:
deals, opportunities, best = run_cycle(RAW_FEED, min_discount=40.0)

print(f"Scanned deals: {len(deals)}")
print("\nRanked opportunities:\n")
for i, opp in enumerate(opportunities, start=1):
    print(
        f"{i}. discount=${opp.discount:7.2f} | conf={opp.confidence:.3f} | "
        f"listed=${opp.deal.listed_price:7.2f} | est=${opp.estimate:7.2f}"
    )
    print(f"   {opp.deal.product_description[:100]}...")

if best:
    print("\nBest opportunity selected by planner:\n")
    print(json.dumps(asdict(best), indent=2))
    notify(best)
else:
    print("\nNo opportunity passed the minimum discount threshold.")

## 6) Optional LLM decision layer (community pattern)

If `OPENAI_API_KEY` exists, this cell asks an LLM to review ranked opportunities and pick one. Otherwise it cleanly skips.

In [ ]:
def llm_choose_best(opportunities: List[Opportunity]) -> Optional[Dict]:
    if not opportunities:
        return None

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("OPENAI_API_KEY not set; skipping LLM decision.")
        return None

    try:
        from openai import OpenAI
    except Exception:
        print("OpenAI SDK not installed; skipping LLM decision.")
        return None

    payload = [
        {
            "id": i,
            "description": opp.deal.product_description,
            "listed_price": opp.deal.listed_price,
            "estimate": opp.estimate,
            "discount": opp.discount,
            "confidence": opp.confidence,
            "url": opp.deal.url,
        }
        for i, opp in enumerate(opportunities, start=1)
    ]

    client = OpenAI(api_key=api_key)
    prompt = (
        "Pick the single best deal from the JSON list. "
        "Return strict JSON with keys: id, rationale.\\n\\n"
        f"{json.dumps(payload)}"
    )

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )

    try:
        return json.loads(response.choices[0].message.content)
    except Exception:
        return {"raw_response": response.choices[0].message.content}


llm_choice = llm_choose_best(opportunities)
print(llm_choice)

## 7) Optional Day 5 mini-UI with Gradio

In [ ]:
try:
    import gradio as gr

    def run_cycle_for_ui():
        _, opps, best_opp = run_cycle(RAW_FEED, min_discount=40.0)
        rows = [
            [
                opp.deal.product_description[:70] + "...",
                f"${opp.deal.listed_price:.2f}",
                f"${opp.estimate:.2f}",
                f"${opp.discount:.2f}",
                f"{opp.confidence:.3f}",
                opp.deal.url,
            ]
            for opp in opps
        ]
        status = "Best deal found and ranked." if best_opp else "No deal over threshold yet."
        return rows, status

    with gr.Blocks(title="Week 8 - Adnan Deal Hunter") as ui:
        gr.Markdown("## Week 8 Deal Hunter (Lite)")
        run_button = gr.Button("Run planner cycle", variant="primary")
        status_box = gr.Textbox(label="Status")
        table = gr.Dataframe(
            headers=["Description", "Listed", "Estimate", "Discount", "Confidence", "URL"],
            wrap=True,
            row_count=5,
        )
        run_button.click(run_cycle_for_ui, outputs=[table, status_box])

    print("UI prepared. Run `ui.launch(inbrowser=True)` when you want to open it.")
except Exception as exc:
    print("Gradio is optional. Install it to use the UI section.")
    print("Details:", exc)

In [ ]:
ui.launch(inbrowser=True)